# Relaxation timing with NVIDIA ALCHEMI, on a platform GPU

Relaxes many strained crystals with MACE through the [NVIDIA ALCHEMI toolkit](https://github.com/NVIDIA/nvalchemi-toolkit)
in three ways, each its own workflow: **ALCHEMI batched** relaxes the structures together in batches on one GPU,
**ALCHEMI serial** and **ASE serial** relax them one at a time. A workflow relaxes the materials of its job and
publishes one timing row as `results.csv`.

This notebook saves the structures as materials in one set, creates the three workflows, runs one job per method
on the whole set, and plots the structures relaxed per second of each method.

Outside the platform's JupyterLite, use Python 3.10 or 3.11 and install the packages once with `pip install "mat3ra-notebooks-utils[all]"`.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime

from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None  # set to use an organization as the owner, otherwise the personal account

# 3. Material parameters
MATERIAL_NAME = "Silicon"  # from Materials Standata
SUPERCELL_SCALING_FACTOR = [2, 2, 2]  # repetitions of the conventional cell
NUMBER_OF_STRUCTURES = 50
RANDOM_SEED = 0
STRAIN_STANDARD_DEVIATION = 0.01  # of each symmetric strain component
STRAIN_LIMIT = 0.02  # strain components are clipped to ±STRAIN_LIMIT
DISPLACEMENT_STANDARD_DEVIATION = 0.03  # Å, of each atomic coordinate
MATERIAL_SET_NAME = "ALCHEMI relaxation structures"

# 4. Workflow parameters
PYTHON_VERSION = "3.11.10"
PYTHON_BUILD = "Default"

# 5. Compute parameters
CLUSTER_NAME = "cluster-003"
QUEUE_NAME = QueueName.GOF  # H100 nodes on cluster-003 (GOF on-demand, GSF spot)
NODES = 1
PPN = 1
TIME_LIMIT = "01:00:00"

# 6. Job parameters: one workflow and one job per method, each job on the whole set
METHODS = ["ALCHEMI batched", "ALCHEMI serial", "ASE serial"]
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()

### 2.3. Select account

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create materials
### 3.1. Build strained and displaced structures

Strained copies of a supercell of the crystal, with every atom displaced at random. Each job relaxes the whole set.

In [ ]:
import numpy as np
from mat3ra.esse.models.core.abstract.matrix_3x3 import Matrix3x3Schema
from mat3ra.made.material import Material
from mat3ra.made.tools.build_components.operations.core.modifications.perturb.helpers import create_maxwell_displacement
from mat3ra.made.tools.build_components.operations.core.modifications.strain.helpers import create_strain
from mat3ra.made.tools.helpers import create_supercell, get_material_with_conventional_lattice
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.standata.materials import Materials

crystal = get_material_with_conventional_lattice(Material.create(Materials.get_by_name_first_match(MATERIAL_NAME)))
supercell = create_supercell(crystal, scaling_factor=SUPERCELL_SCALING_FACTOR)
random_generator = np.random.default_rng(RANDOM_SEED)
materials = []
for index in range(NUMBER_OF_STRUCTURES):
    strain = random_generator.normal(0.0, STRAIN_STANDARD_DEVIATION, size=(3, 3))
    strain = np.clip((strain + strain.T) / 2, -STRAIN_LIMIT, STRAIN_LIMIT)
    strained = create_strain(supercell, strain_matrix=Matrix3x3Schema(root=(np.eye(3) + strain).tolist()))
    material = create_maxwell_displacement(
        strained, disorder_parameter=DISPLACEMENT_STANDARD_DEVIATION**2, random_seed=RANDOM_SEED + index, is_mass_used=False
    )
    material.name = f"{MATERIAL_NAME} {index}"
    materials.append(material)

visualize(materials[0])

### 3.2. Save materials to the platform in one set

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

materials_set = client.materials.create_set({"name": f"{MATERIAL_SET_NAME} {timestamp}", "owner": {"_id": ACCOUNT_ID}})
saved_materials = [get_or_create_material(client, material, ACCOUNT_ID) for material in materials]
for saved_material in saved_materials:
    client.materials.move_to_set(saved_material["_id"], "", materials_set["_id"])

## 4. Create the workflows
### 4.1. Workflow script

The script every workflow runs, with its `METHOD` line set per workflow, and the packages installed for it on the
compute node.

In [ ]:
RELAXATION_SCRIPT = r'''#!/usr/bin/env python3
"""Relaxes the job's materials with METHOD and writes its timing row to results.csv.
Pipeline time excludes model loading, warmup, and file I/O."""

import csv
import json
import sys
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import torch
from ase.calculators.singlepoint import SinglePointCalculator
from ase.filters import FrechetCellFilter
from ase.optimize import FIRE2
from loguru import logger
from mace.calculators import mace_mp
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import to_ase
from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import FIRE2VariableCell, FusedStage, HostMemory, SizeAwareSampler
from nvalchemi.dynamics.base import ConvergenceHook, DynamicsStage
from nvalchemi.models.mace import MACEWrapper

METHOD = "ALCHEMI batched"
DEVICE = "cuda:0"
CHECKPOINT = "medium-mpa-0"
DTYPE = "float32"
FMAX = 0.03  # eV/A, on atomic forces only
MAX_STEPS = 500
TIME_STEP = 0.5
BATCH_SIZE = 10
# The job's materials, written here by the platform when the job is created.
MATERIALS = json.loads(r"""{{ materials | dump | safe }}""")

FIELDS = ("method", "structures", "pipeline_seconds", "structures_per_second",
          "model_load_s", "warmup_s", "input_prepare_s", "optimizer_setup_s",
          "optimize_collect_s", "result_materialization_s")
PIPELINE_FIELDS = ("input_prepare_s", "optimizer_setup_s", "optimize_collect_s",
                   "result_materialization_s")


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


@contextmanager
def timed_phase(phases, name, device):
    synchronize(device)
    start = time.perf_counter()
    try:
        yield
    finally:
        synchronize(device)
        phases[name] = phases.get(name, 0.0) + time.perf_counter() - start


def timed(device, action):
    synchronize(device)
    start = time.perf_counter()
    result = action()
    synchronize(device)
    return result, time.perf_counter() - start


def atomic_data(structures, device, dtype):
    data_list = []
    for i, atoms in enumerate(structures):
        data = AtomicData.from_atoms(atoms, device=device, dtype=dtype)
        data.add_system_property("structure_id", torch.tensor([[i]], device=device,
                                                        dtype=torch.long))
        data["forces"] = torch.zeros_like(data.positions)
        data["energy"] = torch.zeros((1, 1), device=device, dtype=dtype)
        data["stress"] = torch.zeros((1, 3, 3), device=device, dtype=dtype)
        data["velocities"] = torch.zeros_like(data.positions)
        data_list.append(data)
    return data_list


def initialize_runtime_fields(batch):
    batch["forces"] = torch.zeros_like(batch.positions)
    batch["energy"] = torch.zeros((batch.num_graphs, 1), device=batch.device,
                                  dtype=batch.positions.dtype)
    batch["stress"] = torch.zeros((batch.num_graphs, 3, 3), device=batch.device,
                                  dtype=batch.positions.dtype)
    batch["velocities"] = torch.zeros_like(batch.positions)


def alchemi_optimizer(model, device, fmax, steps, dt, fused):
    optimizer = FIRE2VariableCell(
        model=model, dt=dt, n_steps=steps, device_type=device.type,
        convergence_hook=ConvergenceHook(criteria=[{
            "key": "forces", "threshold": fmax, "reduce_op": "norm", "reduce_dims": -1,
        }]),
    )
    if not fused:
        optimizer.register_hook(optimizer.convergence_hook)
        for hook in model.make_neighbor_hooks():
            optimizer.register_hook(hook, stage=DynamicsStage.BEFORE_COMPUTE)
    return optimizer


def log_alchemi_completion(row, mode, steps, fmax):
    maximum = float(torch.linalg.vector_norm(row.forces, dim=-1).max().item())
    logger.info("{} structure_id={} energy={:+.6f} eV fmax={:.6f} eV/A "
                "steps={} status={}", mode, int(row.structure_id.item()),
                float(row.energy.item()), maximum, steps,
                "converged" if maximum < fmax else "max_steps")


class StructureDataset:
    def __init__(self, structures, device, dtype):
        self.data_list = atomic_data(structures, device, dtype)

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, index):
        return self.data_list[index].clone(), {}

    def get_metadata(self, index):
        return self.data_list[index].num_nodes, 0


class CompletionSink(HostMemory):
    def __init__(self, capacity, fmax, max_steps, log_results):
        super().__init__(capacity)
        self.fmax = fmax
        self.max_steps = max_steps
        self.log_results = log_results

    def write(self, batch, mask=None):
        previous = len(self._data_list)
        super().write(batch, mask=mask)
        if self.log_results:
            for row in self._data_list[previous:]:
                counter = getattr(row, "n_steps_counter_0", None)
                steps = int(counter.item()) if counter is not None and counter.item() else self.max_steps
                log_alchemi_completion(row, "inflight", steps, self.fmax)


def relax_inflight(structures, model, device, dtype, fmax, steps, dt, batch_size,
                   *, warmup_only=False):
    phases = {}
    with timed_phase(phases, "input_prepare_s", device):
        dataset = StructureDataset(structures, device, dtype)
    with timed_phase(phases, "optimizer_setup_s", device):
        sampler = SizeAwareSampler(dataset, max_batch_size=batch_size, shuffle=False)
        sink = CompletionSink(len(structures), fmax, steps, not warmup_only)
        optimizer = alchemi_optimizer(model, device, fmax, steps, dt, fused=True)
        stage = FusedStage(sub_stages=[(0, optimizer)], sampler=sampler, sinks=[sink],
                           init_fn=initialize_runtime_fields, device_type=device.type,
                           refill_frequency=1)
        for hook in model.make_neighbor_hooks():
            stage.register_hook(hook, stage=DynamicsStage.BEFORE_COMPUTE)
    with timed_phase(phases, "optimize_collect_s", device):
        remainder = stage.run(n_steps=1 if warmup_only else None)
        result = None if warmup_only else sink.read()
    if not warmup_only and remainder is not None:
        raise RuntimeError("Inflight optimization ended before all results were collected")
    return result, phases


def relax_alchemi_serial(structures, model, device, dtype, fmax, steps, dt,
                         *, warmup_only=False):
    phases = {}
    with timed_phase(phases, "input_prepare_s", device):
        data_list = atomic_data(structures, device, dtype)
    completed = []
    for data in data_list:
        with timed_phase(phases, "optimizer_setup_s", device):
            batch = Batch.from_data_list([data])
            initialize_runtime_fields(batch)
            batch["status"] = torch.zeros((1, 1), device=device, dtype=torch.long)
            optimizer = alchemi_optimizer(model, device, fmax, steps, dt, fused=False)
        with timed_phase(phases, "optimize_collect_s", device):
            optimizer._call_hooks(DynamicsStage.BEFORE_COMPUTE, batch)
            optimizer.compute(batch)
            result = optimizer.run(batch, n_steps=1 if warmup_only else None)
            row = result.to_data_list()[0]
            completed.append(row)
            if not warmup_only:
                log_alchemi_completion(row, "serial", optimizer.step_count, fmax)
    with timed_phase(phases, "optimize_collect_s", device):
        result = Batch.from_data_list(completed)
    return result, phases


def materialize_alchemi(batch, structures, fmax):
    rows = batch.to_data_list()
    if len(rows) != len(structures):
        raise RuntimeError(f"Expected {len(structures)} ALCHEMI results, got {len(rows)}")
    by_id = {int(row.structure_id.item()): row for row in rows}
    if set(by_id) != set(range(len(structures))):
        raise RuntimeError("Missing or duplicate ALCHEMI structure IDs")
    relaxed = []
    for i, original in enumerate(structures):
        row = by_id[i]
        atoms = original.copy()
        atoms.set_cell(row.cell.detach().cpu().numpy().reshape(3, 3), scale_atoms=False)
        atoms.positions = row.positions.detach().cpu().numpy()
        energy = float(row.energy.item())
        forces = row.forces.detach().cpu().numpy()
        maximum = float(np.linalg.norm(forces, axis=-1).max())
        atoms.calc = SinglePointCalculator(atoms, energy=energy, forces=forces)
        if maximum >= fmax:
            raise RuntimeError(f"ALCHEMI structure {i} did not converge: fmax={maximum}")
        relaxed.append(atoms)
    return relaxed


class AtomicForceFIRE2(FIRE2):
    def gradient_converged(self, gradient):
        forces = self.atoms.atoms.get_forces()
        return bool(np.linalg.norm(forces, axis=1).max() < self.fmax)


def ase_optimizer(atoms, dt):
    filtered = FrechetCellFilter(atoms)
    return filtered, AtomicForceFIRE2(filtered, dt=dt, logfile=None)


def relax_ase(structures, calculator, device, fmax, steps, dt):
    phases = dict.fromkeys(PIPELINE_FIELDS, 0.0)
    relaxed = []
    for i, original in enumerate(structures):
        def prepare():
            atoms = original.copy()
            atoms.calc = calculator
            return atoms
        atoms, duration = timed(device, prepare)
        phases["input_prepare_s"] += duration
        (_, optimizer), duration = timed(device, lambda: ase_optimizer(atoms, dt))
        phases["optimizer_setup_s"] += duration

        def optimize_collect():
            converged = optimizer.run(fmax=fmax, steps=steps)
            energy = float(atoms.get_potential_energy())
            forces = atoms.get_forces()
            stress = atoms.get_stress(voigt=False)
            return converged, energy, forces, stress
        (converged, energy, forces, stress), duration = timed(device, optimize_collect)
        phases["optimize_collect_s"] += duration

        def materialize():
            maximum = float(np.linalg.norm(forces, axis=1).max())
            saved = atoms.copy()
            saved.calc = SinglePointCalculator(saved, energy=energy, forces=forces,
                                               stress=stress)
            logger.info("ASE structure_id={} energy={:+.6f} eV fmax={:.6f} eV/A "
                        "steps={} status={}", i, energy, maximum, optimizer.nsteps,
                        "converged" if converged and maximum < fmax else "max_steps")
            if not converged or maximum >= fmax:
                raise RuntimeError(f"ASE structure {i} did not converge: fmax={maximum}")
            return saved
        saved, duration = timed(device, materialize)
        phases["result_materialization_s"] += duration
        relaxed.append(saved)
    return relaxed, phases


def make_row(method, count, model_load, warmup, phases):
    pipeline = sum(phases[key] for key in PIPELINE_FIELDS)
    return {"method": method, "structures": count,
            "pipeline_seconds": pipeline, "structures_per_second": count / pipeline,
            "model_load_s": model_load, "warmup_s": warmup,
            **{key: phases[key] for key in PIPELINE_FIELDS}}


def load_model(device, dtype):
    checkpoint = CHECKPOINT
    if Path(checkpoint).is_file():
        checkpoint = Path(checkpoint).resolve().as_uri()
    with torch.serialization.safe_globals([slice]):
        model = MACEWrapper.from_checkpoint(checkpoint_path=checkpoint,
                                            device=device, dtype=dtype).eval()
    model.model_config.active_outputs = {"energy", "forces", "stress"}
    return model


def run_alchemi_batched(device, structures):
    dtype = getattr(torch, DTYPE)
    model, model_load = timed(device, lambda: load_model(device, dtype))
    _, warmup = timed(device, lambda: relax_inflight(
        structures[:1], model, device, dtype, FMAX, MAX_STEPS, TIME_STEP,
        batch_size=1, warmup_only=True))
    batch, phases = relax_inflight(structures, model, device, dtype,
                                   FMAX, MAX_STEPS, TIME_STEP, BATCH_SIZE)
    _, duration = timed(device, lambda: materialize_alchemi(batch, structures, FMAX))
    phases["result_materialization_s"] = duration
    return model_load, warmup, phases


def run_alchemi_serial(device, structures):
    dtype = getattr(torch, DTYPE)
    model, model_load = timed(device, lambda: load_model(device, dtype))
    _, warmup = timed(device, lambda: relax_alchemi_serial(
        structures[:1], model, device, dtype, FMAX, MAX_STEPS, TIME_STEP,
        warmup_only=True))
    batch, phases = relax_alchemi_serial(structures, model, device, dtype, FMAX,
                                         MAX_STEPS, TIME_STEP)
    _, duration = timed(device, lambda: materialize_alchemi(batch, structures, FMAX))
    phases["result_materialization_s"] = duration
    return model_load, warmup, phases


def run_ase_serial(device, structures):
    calculator, model_load = timed(device, lambda: mace_mp(
        model=CHECKPOINT, device=str(device), default_dtype=DTYPE))

    def ase_warmup():
        atoms = structures[0].copy()
        atoms.positions[0, 0] += 1e-4
        atoms.calc = calculator
        atoms.get_potential_energy()
        atoms.get_forces()
        atoms.get_stress()
        _, optimizer = ase_optimizer(atoms, TIME_STEP)
        optimizer.run(fmax=0.0, steps=1)
    _, warmup = timed(device, ase_warmup)
    _, phases = relax_ase(structures, calculator, device, FMAX, MAX_STEPS, TIME_STEP)
    return model_load, warmup, phases


RUNS = {"ALCHEMI batched": run_alchemi_batched, "ALCHEMI serial": run_alchemi_serial,
        "ASE serial": run_ase_serial}


def main():
    logger.remove()
    logger.add(sys.stderr, level="INFO")
    device = torch.device(DEVICE)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable")
    structures = []
    for document in MATERIALS:
        material = Material.create(document)
        material.to_crystal()
        structures.append(to_ase(material))
    logger.info("{} case: {} structures, {} atoms each, pbc={}", METHOD, len(structures),
                len(structures[0]), structures[0].pbc.tolist())
    model_load, warmup, phases = RUNS[METHOD](device, structures)
    with open("results.csv", "w", newline="") as results_file:
        writer = csv.DictWriter(results_file, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerow(make_row(METHOD, len(structures), model_load, warmup, phases))


if __name__ == "__main__":
    try:
        main()
    except Exception as exc:
        logger.exception("Benchmark failed: {}", exc)
        sys.exit(1)
'''

In [ ]:
REQUIREMENTS = """--find-links https://download.pytorch.org/whl/cu128/torch/
--find-links https://download.pytorch.org/whl/cu128/torchvision/
--find-links https://download.pytorch.org/whl/cu128/torchaudio/
--find-links https://github.com/NVIDIA/warp/releases/expanded_assets/v1.18.0
torch==2.11.0+cu128
torchvision==0.26.0+cu128
torchaudio==2.11.0+cu128
warp-lang==1.18.0+cu12
nvalchemi-toolkit[mace,ase]==0.2.0
matplotlib
numpy
ase
mace-torch
loguru
mat3ra-made[tools]
"""

### 4.2. Create one workflow per method

Each workflow is one step, `relax`. When a job is created, the platform writes the job's materials into the script
(`materials`, as it does for every multi-material workflow), and the script relaxes them with the workflow's method
and publishes its timing row as `results.csv`.

In [ ]:
import copy
import re

from mat3ra.notebooks_utils.core.entity.workflow.api import set_execution_unit_input
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow

custom_script_workflow = WorkflowStandata.filter_by_application("python").get_by_name_first_match("custom_script.json")

workflows = {}
for method in METHODS:
    workflow_config = copy.deepcopy(custom_script_workflow)
    workflow_config.update(name=f"{method} relaxation", isMultiMaterial=True)
    workflow_config["units"][0]["name"] = workflow_config["name"]
    subworkflow = workflow_config["subworkflows"][0]
    relax_unit = subworkflow["units"][-1]
    relax_unit.update(name="relax", head=True, results=[{"name": "file_content", "filetype": "csv", "basename": "results.csv"}])
    for application in (subworkflow["application"], relax_unit["application"]):
        application.update(version=PYTHON_VERSION, build=PYTHON_BUILD)
    script_input = next(unit_input for unit_input in relax_unit["input"] if unit_input["template"]["name"] == "script.py")
    script_input["template"]["content"] = re.sub(r"^METHOD = .*$", f'METHOD = "{method}"', RELAXATION_SCRIPT, flags=re.M)
    set_execution_unit_input(relax_unit, "requirements.txt", REQUIREMENTS)
    subworkflow.update(name=workflow_config["name"], units=[relax_unit])
    workflows[method] = Workflow.create(workflow_config)
    visualize_workflow(workflows[method])

### 4.3. Save workflows to the collection

In [ ]:
for workflow in workflows.values():
    saved_workflow = client.workflows.create(workflow.to_dict_without_special_keys(), owner_id=ACCOUNT_ID)
    print(f"✅ Workflow saved to collection: {saved_workflow['_id']}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[cluster['hostname'] for cluster in clusters]}")

### 5.2. Create compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

cluster = next(cluster for cluster in clusters if CLUSTER_NAME in cluster["hostname"])
compute = Compute(cluster=cluster, queue=QUEUE_NAME, nodes=NODES, ppn=PPN, timeLimit=TIME_LIMIT)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, nodes: {NODES}, ppn: {PPN}, time limit: {TIME_LIMIT}")

## 6. Create one job per method

In [ ]:
from mat3ra.notebooks_utils.job import create_job

job_ids = []
for method in METHODS:
    job = create_job(
        api_client=client,
        materials=saved_materials,
        workflow=workflows[method],
        project_id=project_id,
        owner_id=ACCOUNT_ID,
        prefix=f"{workflows[method].name} {NUMBER_OF_STRUCTURES} {timestamp}",
        compute=compute.to_dict(),
        materials_set=materials_set,
    )
    job_ids.append(job["_id"])
print(f"✅ Created {len(job_ids)} jobs: {job_ids}")

## 7. Submit the jobs and monitor the status

In [ ]:
from mat3ra.notebooks_utils.api.job import submit_jobs

submit_jobs(client.jobs, job_ids)

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, job_ids, poll_interval=POLL_INTERVAL)

## 8. Results
### 8.1. Timing rows of all jobs

In [ ]:
import io

import pandas as pd
from mat3ra.notebooks_utils.io import read_from_url

tables = []
for job_id in job_ids:
    files = {file["key"].rsplit("/", 1)[-1]: file["signedUrl"] for file in client.jobs.list_files(job_id)}
    tables.append(pd.read_csv(io.StringIO(await read_from_url(files["results.csv"]))))
results = pd.concat(tables, ignore_index=True)
results

### 8.2. Structures relaxed per second

In [ ]:
import plotly.express as px

px.line(results.sort_values("structures"), x="structures", y="structures_per_second", color="method", markers=True)